# Workspace check

Run the checks in order using **Python (Fear of Temperature)**, the project `.venv` kernel.
The workspace and synthetic fixture checks must pass before continuing to the synthetic demo.
Optional readiness reports prerequisites for later stages. This notebook reads only the synthetic
fixture and local environment metadata.


In [ ]:
import sys
from importlib import import_module
from pathlib import Path

cwd = Path.cwd().resolve()
project_root = next(
    (
        parent
        for parent in (cwd, *cwd.parents)
        if (parent / "notebooks/01_workspace_check.ipynb").is_file()
        and (parent / "pyproject.toml").is_file()
    ),
    None,
)
if project_root is None:
    raise RuntimeError("Start the notebook in the project workspace or its notebooks directory.")

python_target = (project_root / ".python-version").read_text(encoding="utf-8").strip()
python_version = ".".join(map(str, sys.version_info[:3]))
if python_version != python_target and not python_version.startswith(f"{python_target}."):
    raise RuntimeError(f"Wrong Python version: use the project Python {python_target} kernel.")

if Path(sys.prefix).resolve() != (project_root / ".venv").resolve():
    raise RuntimeError(
        "Wrong Jupyter kernel: select Python (Fear of Temperature) or .venv/bin/python."
    )

package = import_module("fear_temperature")
package_path = Path(package.__file__).resolve()
if package_path != project_root / "src/fear_temperature/__init__.py":
    raise RuntimeError("The imported fear_temperature package is not from this checkout.")
project_config = import_module("tomllib").loads(
    (project_root / "pyproject.toml").read_text(encoding="utf-8")
)
expected_version = project_config["project"]["version"]
if package.__version__ != expected_version:
    raise RuntimeError(
        f"Package version mismatch: expected {expected_version}, found {package.__version__}."
    )

{
    "workspace": "PASS",
    "python": python_version,
    "kernel_environment": ".venv",
    "package": package_path.relative_to(project_root).as_posix(),
    "package_version": package.__version__,
}

In [ ]:
from datetime import date
from hashlib import sha256

from fear_temperature.cleaning import VALID_ROLES, prepare_corpus
from fear_temperature.config import load_yaml
from fear_temperature.io import read_records

FIXTURE = Path("data/fixtures/synthetic_passages.jsonl")
FIXTURE_SHA256 = "9379653ca6b2240b96d8c3e4a53d0f23b72e4300199cd2a3737dae790ff068ce"
REQUIRED_TEXT = ("external_id", "source_name", "publisher_role", "publication_date", "text")
BOOLEAN_FIELDS = ("fixture_relevant", "fixture_future_worry")


def validate_fixture(path: Path) -> dict:
    if not path.is_file():
        raise ValueError(f"Synthetic fixture missing: {path.name}.")
    digest = sha256(path.read_bytes()).hexdigest()
    if digest != FIXTURE_SHA256:
        raise ValueError(f"Synthetic fixture SHA-256 mismatch: {path.name}.")
    records = read_records(path)
    for row, record in enumerate(records, start=1):
        for field in REQUIRED_TEXT:
            value = record.get(field)
            if not isinstance(value, str) or not value.strip():
                raise ValueError(f"Synthetic fixture row {row}: {field} requires non-empty text.")
        for field in BOOLEAN_FIELDS:
            if not isinstance(record.get(field), bool):
                raise ValueError(f"Synthetic fixture row {row}: {field} requires a JSON boolean.")
        if record["publisher_role"] not in VALID_ROLES:
            raise ValueError(f"Synthetic fixture row {row}: invalid publisher role.")
        try:
            publication_date = date.fromisoformat(record["publication_date"])
        except ValueError as error:
            raise ValueError(f"Synthetic fixture row {row}: invalid publication date.") from error
        if publication_date.isoformat() != record["publication_date"]:
            raise ValueError(f"Synthetic fixture row {row}: expected YYYY-MM-DD.")

    corpus = prepare_corpus(records)
    if sha256(path.read_bytes()).hexdigest() != digest:
        raise ValueError("Synthetic fixture changed during the check.")
    counts = (
        len(records),
        len(corpus.lineage),
        len(corpus.passages),
        int(corpus.lineage["is_exact_duplicate"].sum()),
    )
    if counts != (37, 37, 36, 1):
        raise ValueError(
            f"Synthetic fixture count mismatch (input, lineage, canonical, duplicate): {counts}."
        )
    roles = set(corpus.passages["publisher_role"])
    if roles != VALID_ROLES:
        raise ValueError(f"Synthetic fixture role mismatch: {sorted(roles)}.")
    return {
        "synthetic_fixture": "PASS",
        "sha256": digest,
        "input_records": counts[0],
        "lineage_rows": counts[1],
        "canonical_passages": counts[2],
        "exact_duplicates": counts[3],
        "roles": sorted(roles),
    }


if load_yaml("configs/default.yaml")["demo"]["input"] != FIXTURE.as_posix():
    raise RuntimeError("The default demo input does not match the checked synthetic fixture.")

fixture_result = validate_fixture(project_root / FIXTURE)
fixture_result

In [ ]:
# Optional readiness for the offline model demo and local Spark notebooks.
from importlib.metadata import PackageNotFoundError, version
from shutil import which
from subprocess import TimeoutExpired, run

from fear_temperature.config import Settings, resolve_device


def installed_version(distribution: str) -> str | None:
    try:
        return version(distribution)
    except PackageNotFoundError:
        return None


settings = Settings.from_env()
models = load_yaml("configs/models.yaml")
java = which("java")
java_status = "Java launcher unavailable"
if java:
    try:
        probe = run([java, "-version"], capture_output=True, text=True, timeout=5, check=False)
        if probe.returncode == 0:
            version_lines = (probe.stderr or probe.stdout).strip().splitlines()
            java_status = version_lines[0] if version_lines else "Java returned no version text"
        else:
            java_status = f"Java failed to start (exit {probe.returncode})"
    except (OSError, TimeoutExpired):
        java_status = "Java probe failed or timed out"

offline_snapshots = {}
for name in ("sentence_encoder", "emotion_candidate"):
    model = models[name]
    repository = f"models--{model['id'].replace('/', '--')}"
    snapshot = settings.model_cache / repository / "snapshots" / model["revision"]
    offline_snapshots[name] = (
        "Snapshot directory found" if snapshot.is_dir() else "Snapshot missing"
    )

device, device_details = resolve_device(settings.device_preference)
{
    "spark_prerequisites": {"pyspark_version": installed_version("pyspark"), "java": java_status},
    "offline_model_cache": offline_snapshots,
    "installed_parser_version": installed_version(models["parser"]["id"]),
    "device_probe": {"selected": device, **device_details},
    "scope": "No Spark session started. Model files and inference were not checked.",
}